# CPE extraction from certificate titles: Colab notebook
Runtime → Change runtime type → **T4 GPU**. Run only the cells listed for the stage you are working on.

**Cells 1-7: Phase 2, Yasir's baseline on the gold set (done; kept as a trace of the work).** Outputs were cleared because they showed an older gold set. The final baseline numbers are in the project report (F1 0.44).

**Cells 8-17: Phase 5, train our own adapter v1 on `data/synthetic_v2` and test it on the same gold set.**
Run for this stage, in order: **1, 8, 9, 10, 11, 12, 14, 15, 16, 17**. (Cell 13 only after a runtime reset. Skip 2-7.)

In [ ]:
# WHAT IT DOES: clones the project repo (or pulls the latest commit) and installs dependencies. Run it first in every new session.
# PHASE: shared by all stages.
# 1. Clone (or update) the repo and install dependencies
REPO_URL = "https://github.com/m-mirkan/peft-cpe-extraction-pipeline.git"
BRANCH = "feat/codebook-data-curation"

import os
if not os.path.exists("/content/repo"):
    !git clone -b {BRANCH} {REPO_URL} /content/repo
%cd /content/repo
!git fetch origin
!git checkout {BRANCH}
!git pull origin {BRANCH}
!pip uninstall -y -q torchao
!pip install -q -r requirements.txt gdown

import torch
print("CUDA available:", torch.cuda.is_available(), "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
# WHAT IT DOES: downloads Yasir's baseline LoRA adapter from Google Drive into checkpoints/qwen25_3b_cpe_lora.
# PHASE 2 (baseline) only. NOT needed for training our own adapter.
# 2. Download baseline adapter
import os, zipfile
LORA_GDRIVE_ID = "1eKyqTdDT36zPEpi6y62kZg9V8vTmJVaX"
LORA_DIR = "checkpoints/qwen25_3b_cpe_lora"
os.makedirs(LORA_DIR, exist_ok=True)

if not os.path.exists(os.path.join(LORA_DIR, "adapter_config.json")):
    !gdown {LORA_GDRIVE_ID} -O lora_adapter.zip
    with zipfile.ZipFile("lora_adapter.zip", "r") as z:
        z.extractall(LORA_DIR)
    entries = os.listdir(LORA_DIR)
    if len(entries) == 1 and os.path.isdir(os.path.join(LORA_DIR, entries[0])):
        nested = os.path.join(LORA_DIR, entries[0])
        for f in os.listdir(nested):
            os.rename(os.path.join(nested, f), os.path.join(LORA_DIR, f))
        os.rmdir(nested)
print("LoRA adapter files:", os.listdir(LORA_DIR))

In [ ]:
# WHAT IT DOES: checks that the gold files and the patched inference script are in the repo.
# PHASE 2 (baseline). Safe to run again, takes a second.
# 3. Sanity check: the gold files and the patched script must be there
import json
!ls data/gold scripts
inputs = json.load(open("data/gold/gold_inputs.json", encoding="utf-8"))
gold = json.load(open("data/gold/gold_template.json", encoding="utf-8"))
print("gold titles:", len(inputs), "| unique:", len(set(inputs)), "| labeled:", sum(g["status"] == "labeled" for g in gold))
assert len(inputs) == 100 and [g["input"] for g in gold] == inputs
print("patched inference script (should print 2 lines):")
!grep -n "max_new_tokens=2048\|FAILED" src/infer_and_resolve.py

In [ ]:
# WHAT IT DOES: runs Yasir's baseline adapter on gold titles 1-50 and saves predictions after every title.
# PHASE 2 (baseline), already done. Do NOT rerun for the new stage.
# 4. Run titles 1-50 (saves after every title; failures are recorded, not fatal)
!python src/infer_and_resolve.py --lora_dir checkpoints/qwen25_3b_cpe_lora --input_file data/gold/gold_inputs.json --offset 0 --limit 50 --output_file outputs/gold_base_p1.json

In [ ]:
# WHAT IT DOES: runs Yasir's baseline adapter on gold titles 51-100.
# PHASE 2 (baseline), already done. Do NOT rerun for the new stage.
# 5. Run titles 51-100
!python src/infer_and_resolve.py --lora_dir checkpoints/qwen25_3b_cpe_lora --input_file data/gold/gold_inputs.json --offset 50 --limit 50 --output_file outputs/gold_base_p2.json

In [ ]:
# WHAT IT DOES: scores the baseline predictions against the gold set (precision, recall, F1, part accuracy, exact title).
# PHASE 2 (baseline), already done. Final baseline result: F1 0.44 (see the report).
# 6. Score against the gold set
import json
n1 = len(json.load(open("outputs/gold_base_p1.json", encoding="utf-8")))
n2 = len(json.load(open("outputs/gold_base_p2.json", encoding="utf-8")))
print("predictions saved:", n1, "+", n2, "(expected 50 + 50)")
!python scripts/score_gold.py --gold data/gold/gold_template.json --pred baseline=outputs/gold_base_p1.json,outputs/gold_base_p2.json --json_out outputs/gold_base_scores.json

In [ ]:
# WHAT IT DOES: downloads the baseline prediction and score files to your computer.
# PHASE 2 (baseline), already done.
# 7. Download the outputs
from google.colab import files
files.download("outputs/gold_base_p1.json")
files.download("outputs/gold_base_p2.json")
files.download("outputs/gold_base_scores.json")

## Phase 5: train our own adapter (v1) and test it on the gold set
Our data: `data/synthetic_v2/` (codebook rules, zero overlap with the gold set). Yasir's data and adapter are never touched. Our adapter goes to `checkpoints/qwen25_3b_cpe_lora_v1`.

In [ ]:
# 8. WHAT IT DOES: mounts Google Drive and defines the project folders where adapters and results are kept between sessions.
# Layout on Drive:  MyDrive/lora_projects/cpe_certificate_extraction/{adapters, results}
# (lora_projects/ can hold other LoRA projects later; each project gets its own subfolder.)
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PROJECT = "/content/drive/MyDrive/lora_projects/cpe_certificate_extraction"
DRIVE_ADAPTERS = DRIVE_PROJECT + "/adapters"
DRIVE_RESULTS = DRIVE_PROJECT + "/results"
!mkdir -p "{DRIVE_ADAPTERS}" "{DRIVE_RESULTS}"
print("Drive folders ready:", DRIVE_ADAPTERS, "|", DRIVE_RESULTS)

In [ ]:
# 9. WHAT IT DOES: sanity check before training. Confirms our v2 training files are in the repo (2200 train / 200 val),
# that they differ from Yasir's folder, and that the gold files are present.
import os, json
for p in ["data/synthetic_v2/synthetic_train.jsonl", "data/synthetic_v2/synthetic_val.jsonl",
          "data/gold/gold_inputs.json", "data/gold/gold_template.json"]:
    assert os.path.exists(p), "MISSING (did you push and pull?): " + p
n_tr = sum(1 for _ in open("data/synthetic_v2/synthetic_train.jsonl", encoding="utf-8"))
n_va = sum(1 for _ in open("data/synthetic_v2/synthetic_val.jsonl", encoding="utf-8"))
print("v2 train:", n_tr, "| v2 val:", n_va, "(expected 2200 / 200)")
assert (n_tr, n_va) == (2200, 200)
print("first v2 record input:", json.loads(open("data/synthetic_v2/synthetic_train.jsonl", encoding="utf-8").readline())["input"])
print("OK. Training output folder will be checkpoints/qwen25_3b_cpe_lora_v1 (Yasir's folder is checkpoints/qwen25_3b_cpe_lora).")

In [ ]:
# 10. WHAT IT DOES: trains our own LoRA adapter (v1) on the v2 data. Same defaults as Yasir's training
# (2 epochs, lr 2e-4, LoRA r=16) so the comparison is fair. Saves to a NEW folder; the adapter is written only at the very end.
# Takes a while on a T4 (can be over an hour): keep the tab open. Paste the last train/validation loss lines to Claude.
!python src/train_qwen.py \
  --train_file data/synthetic_v2/synthetic_train.jsonl \
  --val_file data/synthetic_v2/synthetic_val.jsonl \
  --output_dir checkpoints/qwen25_3b_cpe_lora_v1

In [ ]:
# 11. WHAT IT DOES: copies our trained adapter to Google Drive right away (Colab storage is wiped when the runtime ends)
# and also makes a zip, like Yasir's, that can be shared with him.
import os
assert os.path.exists("checkpoints/qwen25_3b_cpe_lora_v1/adapter_config.json"), "adapter not found: did training (cell 10) finish?"
!cp -r checkpoints/qwen25_3b_cpe_lora_v1 "{DRIVE_ADAPTERS}/"
!cd checkpoints && zip -r -q "{DRIVE_ADAPTERS}/qwen25_3b_cpe_lora_v1.zip" qwen25_3b_cpe_lora_v1
print("Saved on Drive:")
!ls -la "{DRIVE_ADAPTERS}"

In [ ]:
# 12. WHAT IT DOES: lists the adapter files to confirm the adapter is complete (adapter_model.safetensors + adapter_config.json).
print(sorted(os.listdir("checkpoints/qwen25_3b_cpe_lora_v1")))

In [ ]:
# 13. WHAT IT DOES: RESTORES our adapter from Drive. Run this ONLY in a new session / after a runtime reset
# (run cells 1 and 8 first). Skips the copy if the adapter is already in checkpoints/.
import os
if not os.path.exists("checkpoints/qwen25_3b_cpe_lora_v1/adapter_config.json"):
    os.makedirs("checkpoints", exist_ok=True)
    !cp -r "{DRIVE_ADAPTERS}/qwen25_3b_cpe_lora_v1" checkpoints/
print(sorted(os.listdir("checkpoints/qwen25_3b_cpe_lora_v1")))

In [ ]:
# 14. WHAT IT DOES: runs OUR adapter (v1) on gold titles 1-50. Predictions are saved after every title
# to outputs/gold_v1_p1.json (new file, the baseline files are not overwritten).
!python src/infer_and_resolve.py --lora_dir checkpoints/qwen25_3b_cpe_lora_v1 --input_file data/gold/gold_inputs.json --offset 0 --limit 50 --output_file outputs/gold_v1_p1.json

In [ ]:
# 15. WHAT IT DOES: runs OUR adapter (v1) on gold titles 51-100 -> outputs/gold_v1_p2.json.
!python src/infer_and_resolve.py --lora_dir checkpoints/qwen25_3b_cpe_lora_v1 --input_file data/gold/gold_inputs.json --offset 50 --limit 50 --output_file outputs/gold_v1_p2.json

In [ ]:
# 16. WHAT IT DOES: scores our v1 adapter on the gold set and, if the baseline files are in the repo
# (results/baseline_v0/), prints both models side by side (same scorer, same 100 titles).
import json, os
n1 = len(json.load(open("outputs/gold_v1_p1.json", encoding="utf-8")))
n2 = len(json.load(open("outputs/gold_v1_p2.json", encoding="utf-8")))
print("v1 predictions saved:", n1, "+", n2, "(expected 50 + 50)")
cmd = "python scripts/score_gold.py --gold data/gold/gold_template.json --pred v1=outputs/gold_v1_p1.json,outputs/gold_v1_p2.json"
b1, b2 = "results/baseline_v0/gold_base_p1.json", "results/baseline_v0/gold_base_p2.json"
if os.path.exists(b1) and os.path.exists(b2):
    cmd += f" --pred baseline={b1},{b2}"
else:
    print("baseline files not found in results/baseline_v0/, scoring v1 only")
cmd += " --json_out outputs/gold_v1_scores.json"
!{cmd}

In [ ]:
# 17. WHAT IT DOES: backs up the v1 predictions and scores to Drive, then downloads them to your computer.
# Send gold_v1_p1.json and gold_v1_p2.json (and the scores file) to Claude for the comparison and the report.
!cp outputs/gold_v1_p1.json outputs/gold_v1_p2.json outputs/gold_v1_scores.json "{DRIVE_RESULTS}/"
from google.colab import files
for f in ["outputs/gold_v1_p1.json", "outputs/gold_v1_p2.json", "outputs/gold_v1_scores.json"]:
    files.download(f)